In [1]:
import numpy as np
from pathlib import Path

processed_dir = Path("../data/processed")

X_train = np.load(processed_dir / "X_train_scaled.npy")
X_test = np.load(processed_dir / "X_test_scaled.npy")
y_train = np.load(processed_dir / "y_train.npy")
y_test = np.load(processed_dir / "y_test.npy")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining labels:")
print("Normal:", (y_train == 0).sum())
print("Attack:", (y_train == 1).sum())

X_train: (175341, 194)
X_test: (82332, 194)
y_train: (175341,)
y_test: (82332,)

Training labels:
Normal: 56000
Attack: 119341


In [2]:
from xgboost import XGBClassifier
import time

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

start = time.time()

model.fit(X_train, y_train)

end = time.time()

print("Training complete!")
print(f"Training time: {end - start:.2f} seconds")

Training complete!
Training time: 2.61 seconds


In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.8729534081523588
Precision: 0.8217805337171963
Recall   : 0.9822862437130504
F1 Score : 0.8948933861211037
ROC-AUC  : 0.9831144332146493

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.74      0.84     37000
           1       0.82      0.98      0.89     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.87      0.87     82332


Confusion Matrix:
[[27343  9657]
 [  803 44529]]


In [4]:
from xgboost import XGBClassifier
import time

gpu_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    tree_method="hist",
    device="cuda"
)

start = time.time()

gpu_model.fit(X_train, y_train)

end = time.time()

print("GPU training complete!")
print(f"GPU training time: {end - start:.2f} seconds")

C:\Users\hsv89\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:38:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\hsv89\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:38:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


GPU training complete!
GPU training time: 2.82 seconds


In [5]:
import joblib
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, models_dir / "xgboost_baseline.joblib")

print("Saved:", models_dir / "xgboost_baseline.joblib")

Saved: ..\models\xgboost_baseline.joblib
